# Recuperación: BM25 frente a búsqueda densa

BM25 cuenta palabras con saturación y normalización por largo. La búsqueda densa compara vectores. Medimos ambas con recall@k y NDCG@k sobre un mini-corpus con relevancia marcada a mano.

In [1]:
import numpy as np

from common import set_seed, setup_style

setup_style()
rng = set_seed(42)

In [2]:
DOCS = {
    "d1": "introducción al aprendizaje por refuerzo: agentes, recompensas y políticas",
    "d2": "q-learning profundo para juegos de atari",
    "d3": "revisión de métodos de gradiente de política",
    "d4": "enfermedades del cacao en colombia",
    "d5": "procesos de decisión de markov y programación dinámica",
    "d6": "redes neuronales convolucionales para imágenes",
    "d7": "tutorial de aprendizaje por refuerzo para principiantes",
    "d8": "series de tiempo y pronóstico de demanda",
}
RELEVANT = {"d1", "d3", "d5", "d7"}
QUERY = "aprendizaje por refuerzo introducción"

In [3]:
from research_paths.evaluation import ndcg_at_k, recall_at_k
from research_paths.retrieval import BM25, cosine_scores, hashing_encoder, simple_tokenize

ids, texts = list(DOCS), list(DOCS.values())
bm = BM25().fit([simple_tokenize(t) for t in texts]).scores(simple_tokenize(QUERY))
enc = hashing_encoder(512)
dense = cosine_scores(enc([QUERY], "query")[0], enc(texts, "passage"))
for name, s in [("BM25", bm), ("denso (hashing)", dense)]:
    ranked = [ids[i] for i in np.argsort(-s, kind="mergesort")]
    r, n = recall_at_k(ranked, RELEVANT, 4), ndcg_at_k(ranked, RELEVANT, 4)
    print(f"{name:16s} {ranked[:4]}  recall@4={r:.2f}  NDCG@4={n:.2f}")

BM25             ['d1', 'd7', 'd2', 'd3']  recall@4=0.75  NDCG@4=0.80
denso (hashing)  ['d1', 'd7', 'd2', 'd3']  recall@4=0.75  NDCG@4=0.80


El codificador de hashing es una bolsa de palabras: no entiende que «procesos de decisión de Markov» es parte del tema. e5 (notebook 06 y el proyecto) sí lo capta, porque aprendió de millones de pares de textos relacionados. BM25 sigue siendo una línea base difícil de batir cuando la consulta comparte palabras con los documentos.